In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

credit_df = pd.read_csv("../data/raw/loan_approval.csv")
credit_df.head()

,loan_id,no_of_dependents,education,self_employed,income_annum,loan_amount,loan_term,cibil_score,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value,loan_status
0,1,2,Graduate,No,9600000,29900000,12,778,2400000,17600000,22700000,8000000,Approved
1,2,0,Not Graduate,Yes,4100000,12200000,8,417,2700000,2200000,8800000,3300000,Rejected
2,3,3,Graduate,No,9100000,29700000,20,506,7100000,4500000,33300000,12800000,Rejected
3,4,3,Graduate,No,8200000,30700000,8,467,18200000,3300000,23300000,7900000,Rejected
4,5,5,Not Graduate,Yes,9800000,24200000,20,382,12400000,8200000,29400000,5000000,Rejected


In [2]:
# Dimensiones del dataset
credit_df.shape

(4269, 13)

In [3]:
#Eliminamos la columa loan_id que no aporta señal
credit_df = credit_df.drop(columns=['loan_id'])


In [4]:
credit_df.columns = credit_df.columns.str.strip().str.lower()#Limpiamos los espacios



credit_df['residential_assets_value'] = (
    credit_df['residential_assets_value'].clip(lower=0)
)
# Corrección de valores negativos en activos residenciales
# En la realidad, un activo no puede tener valor negativo.
# Se asume que valores negativos son errores de carga o placeholders.

In [5]:
print(credit_df['loan_status'].head())

0     Approved
1     Rejected
2     Rejected
3     Rejected
4     Rejected
Name: loan_status, dtype: object


In [6]:
credit_df['loan_status'].value_counts()

 Approved    2656
 Rejected    1613
Name: loan_status, dtype: int64

In [7]:
credit_df.dtypes

no_of_dependents             int64
education                   object
self_employed               object
income_annum                 int64
loan_amount                  int64
loan_term                    int64
cibil_score                  int64
residential_assets_value     int64
commercial_assets_value      int64
luxury_assets_value          int64
bank_asset_value             int64
loan_status                 object
dtype: object

In [8]:
# Mapear la variable target a valores numéricos
# Approved → 1, Rejected → 0
credit_df['loan_status'] = credit_df['loan_status'].str.strip().str.capitalize()
credit_df['loan_status'] = credit_df['loan_status'].map({'Approved':1,'Rejected':0})
print(credit_df['loan_status'].value_counts())



1    2656
0    1613
Name: loan_status, dtype: int64


**Dataset ligeramente desbalanceado**

In [9]:
print(credit_df.describe(include='all'))

        no_of_dependents  education self_employed  income_annum   loan_amount  \
count        4269.000000       4269          4269  4.269000e+03  4.269000e+03   
unique               NaN          2             2           NaN           NaN   
top                  NaN   Graduate           Yes           NaN           NaN   
freq                 NaN       2144          2150           NaN           NaN   
mean            2.498712        NaN           NaN  5.059124e+06  1.513345e+07   
std             1.695910        NaN           NaN  2.806840e+06  9.043363e+06   
min             0.000000        NaN           NaN  2.000000e+05  3.000000e+05   
25%             1.000000        NaN           NaN  2.700000e+06  7.700000e+06   
50%             3.000000        NaN           NaN  5.100000e+06  1.450000e+07   
75%             4.000000        NaN           NaN  7.500000e+06  2.150000e+07   
max             5.000000        NaN           NaN  9.900000e+06  3.950000e+07   

          loan_term  cibil_

In [10]:
credit_df.isnull().sum() #Missing values 

no_of_dependents            0
education                   0
self_employed               0
income_annum                0
loan_amount                 0
loan_term                   0
cibil_score                 0
residential_assets_value    0
commercial_assets_value     0
luxury_assets_value         0
bank_asset_value            0
loan_status                 0
dtype: int64

In [11]:
credit_df['loan_status'] #Chequeo final del target

0       1
1       0
2       0
3       0
4       0
       ..
4264    0
4265    1
4266    0
4267    1
4268    1
Name: loan_status, Length: 4269, dtype: int64

In [12]:
#Lista de variables categóricas
cat_cols = ['education', 'self_employed']

#Crear variables dummy (0/1) para cada categoría
credit_df = pd.get_dummies(credit_df, columns=cat_cols, drop_first=True)

print(credit_df.head())

   no_of_dependents  income_annum  loan_amount  loan_term  cibil_score  \
0                 2       9600000     29900000         12          778   
1                 0       4100000     12200000          8          417   
2                 3       9100000     29700000         20          506   
3                 3       8200000     30700000          8          467   
4                 5       9800000     24200000         20          382   

   residential_assets_value  commercial_assets_value  luxury_assets_value  \
0                   2400000                 17600000             22700000   
1                   2700000                  2200000              8800000   
2                   7100000                  4500000             33300000   
3                  18200000                  3300000             23300000   
4                  12400000                  8200000             29400000   

   bank_asset_value  loan_status  education_ Not Graduate  self_employed_ Yes  
0           

In [13]:
print(credit_df.columns)

Index(['no_of_dependents', 'income_annum', 'loan_amount', 'loan_term',
       'cibil_score', 'residential_assets_value', 'commercial_assets_value',
       'luxury_assets_value', 'bank_asset_value', 'loan_status',
       'education_ Not Graduate', 'self_employed_ Yes'],
      dtype='object')


In [14]:
# Scalar los datos, (solo variables continuas)

num_cols_scale = [
    'no_of_dependents',
    'income_annum',
    'loan_amount',
    'loan_term',
    'cibil_score',
    'residential_assets_value',
    'commercial_assets_value',
    'luxury_assets_value',
    'bank_asset_value'
]

#'loan_status' queda afuera porque nunca se escala la variable target

In [15]:
scaler = StandardScaler()
credit_df[num_cols_scale] = scaler.fit_transform(credit_df[num_cols_scale])
credit_df.head()


,no_of_dependents,income_annum,loan_amount,loan_term,cibil_score,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value,loan_status,education_ Not Graduate,self_employed_ Yes
0,-0.294102,1.617979,1.633052,0.192617,1.032792,-0.780249,2.877289,0.832028,0.930304,1,0,0
1,-1.473548,-0.341750,-0.324414,-0.508091,-1.061051,-0.734111,-0.631921,-0.694993,-0.515936,0,1,1
2,0.295621,1.439822,1.610933,1.594031,-0.544840,-0.057408,-0.107818,1.996520,2.407316,0,0,0
3,0.295621,1.119139,1.721525,-0.508091,-0.771045,1.649729,-0.381263,0.897943,0.899533,0,0,0
4,1.475067,1.689242,1.002681,1.594031,-1.264055,0.757711,0.735304,1.568075,0.007172,0,1,1


In [17]:
# Guardar dataset procesado para modelado

credit_df.to_csv(
    "../data/processed/credit_df_processed.csv",
    index=False
)
